# Experiments

In [1]:
import pickle
import ffmpeg
import os

In [2]:
with open("/home/jobin/Projects/transcript_summarizer/data/pickled/result_list.pkl", "rb") as f:
    res_list = pickle.load(f)

/home/jobin/Projects/transcript_summarizer/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
res_list

[{'query': 'Focuses on deep work sessions in two four-hour blocks of uninterrupted time',
  'retrieved': [Document(id=aa3504574bceb84836cfcb614c2d546161dfc816e35b4093b75302f77820a69b, content: 'So I'm going to eat just one meal today, which will be after this four-hour session and I see this a...', meta: {'start': 1044.94, 'end': 1101.14, 'words': [[{'word': ' So', 'start': 1044.94, 'end': 1045.02, 'probability': 0.8684313893318176, 'tokens': [407]}, {'word': " I'm", 'start': 1045.02, 'end': 1045.14, 'probability': 0.9024690091609955, 'tokens': [286, 478]}, {'word': ' going', 'start': 1045.14, 'end': 1045.28, 'probability': 0.9954349398612976, 'tokens': [516]}, {'word': ' to', 'start': 1045.4, 'end': 1045.5, 'probability': 0.9962933659553528, 'tokens': [281]}, {'word': ' eat', 'start': 1045.5, 'end': 1045.82, 'probability': 0.9922882914543152, 'tokens': [1862]}, {'word': ' just', 'start': 1045.82, 'end': 1046.14, 'probability': 0.981792151927948, 'tokens': [445]}, {'word': ' one', 'sta

In [4]:
for res_dict in res_list:
    for doc in res_dict["retrieved"]:
        print(doc.meta["start"], doc.meta["end"])
    print("#################")

1044.94 1101.14
1607.36 1656.18
#################
1031.46 1037.76
1037.76 1044.34
#################
1204.3 1301.22
1464.46 1494.82
#################
1336.14 1361.84
#################


In [5]:
TMP_FOLDER = "/home/jobin/Projects/transcript_summarizer/data/tmp"

In [14]:
def generate_video_trims(video_path: str, timestamps: list[tuple[float]], save_folder: str) -> str:
    os.makedirs(save_folder, exist_ok=True)
    trim_file_list_path = f'{save_folder}/trim_file_list.txt'
    temp_files = [f'{save_folder}/trim_{i}.mp4' for i in range(len(timestamps))]

    # Trim each segment
    for i, (start, end) in enumerate(timestamps):
        (
            ffmpeg
            .input(video_path, ss=start, to=end)
            .output(temp_files[i], codec='copy')
            .run(overwrite_output=True)
        )

    # Concatenate the trimmed segments
    with open(trim_file_list_path, 'w') as f:
        for temp_file in temp_files:
            f.write(f"file '{temp_file}'\n")
    return trim_file_list_path

def merge_trim_files(trim_file_list_path: str, save_folder: str, save_name: str) -> str:
    merged_video_path = f'{save_folder}/{save_name}'
    ffmpeg.input(trim_file_list_path, format='concat', safe=0).output(merged_video_path, c='copy').run()
    return merged_video_path

def cleanup(folder_path: str) -> None:
    # Clean up temporary files
    for f in os.listdir(folder_path):
        os.remove(f'{folder_path}/{f}')

def generate_merged_video_for_summary_bullets(video_path: str, summary_dict: dict[str], merged_file_idx: int, trims_save_folder: str, merged_files_save_folder: str) -> str:
    filename = video_path.split("/")[-1].split(".")[0]
    timestamps = [] 
    for doc in summary_dict["retrieved"]:
        timestamps.append((doc.meta["start"], doc.meta["end"]))
    trim_file_list_path = generate_video_trims(video_path, timestamps, trims_save_folder)
    merged_video_path = merge_trim_files(trim_file_list_path, merged_files_save_folder, f'{filename}_{merged_file_idx}.mp4')
    cleanup(trims_save_folder)
    return merged_video_path
    

In [12]:
video_path = "/home/jobin/Projects/transcript_summarizer/data/fridman_day.mp4"

for i, res_dict in enumerate(res_list):
    merged_video_path = generate_merged_video_for_summary_bullets(video_path, res_dict, suffix=i)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab